# 1. 주제
**: 버섯의 특징에 따른 독성(p), 식용(e) 분류 문제**

# 2. 데이터
**- train.csv**
- id: 샘플별 고유 ID
- class: 타겟 변수. 'e=식용, p=독버섯'으로 구성
- 버섯 특징 데이터(20개 컬럼): cap-diameter, cap-shape, habitat, season 등

**- test.csv**
- id: 샘플별 고유 ID
- 버섯 특징 데이터(20개 컬럼): cap-diameter, cap-shape, habitat, season 등

**- sample_submission.csv**
- id: 샘플별 고유 ID
- class: 예측한 버섯 분류 결과(e 또는 p)

# 3. 코드 흐름
**1. 데이터 정보 확인**
- 전체적인 데이터 정보 확인: info()함수를 통해 데이터의 총 개수와 columns, 각 column의 유형(dtype)을 확인
- isnull().sum(): 결측치 파악: isnull().sum() 함수를 기반으로 결측치의 개수와 결측치의 비율을 확인. veil-type, spore-print-color, stem-root 등 9개의 칼럼은 결측치가 100,000개 이상(10% ~ 90% 이상)임을 확인
- 타겟 분포 확인: check_target_dist() 함수를 통해 피처의 범주별(season, habitat)로 독버섯일 확률을 계산 (예: 가을(a)과 여름(u)에 독버섯 비율이 상대적으로 높음)


**2. 데이터 전처리**
- p=1, e=0 표준화: map()함수를 통해 독버섯(p)를 1, 식용버섯(e)를 0으로 표준화
- Label Encoding: 모든 범주형 변수를 숫자로 변환하되, encoders라는 딕셔너리에 각 컬럼의 인코더를 저장
- Unknown 강제 학습: 학습 데이터에 없던 새로운 값이 테스트 데이터에 등장할 경우를 대비해 "unknown"을 학습 및 추가

**3. 모델링 및 평가**
- 데이터 분할: train_test_split() 함수를 통해 20:80의 비율(test_size = 0.2)로 train/test data 분할
- LightGBM: LightGBM 모델로 예측 및 평가 진행
- 평가 지표: 단순히 Accuracy만 보지 않고 MCC 점수를 확인하고, confusion_matrix()함수를 통해 혼동행렬까지 확인. classification_report() 함수를 통해 precision, recall, f1-score, support, accuracy, macro avg, weighted avg가 모두 포함되어 있는 LightGBM 상세 보고서를 출력함

**4. 예측 및 제출 파일 생성**
- unknown 대체: apply lambda 함수를 활용하여 훈련 시 없던 값은 "unknown"으로 대체시킴
- 제출 파일 생성: sample_submission 파일에 데이터를 저장시키고 분석 마무리

# 3-1. 주요 코드

In [ ]:
# 1. 피처의 범주별로 독버섯(p)과 식용(e)의 비율을 계산 (백분율)

def check_target_dist(df, feature):
    dist = pd.crosstab(df[feature], df['class'], normalize='index') * 100
    dist['total_count'] = df[feature].value_counts()
    return dist.sort_values(by='p', ascending=False) # 독버섯 비율 높은 순 정렬

In [ ]:
# 2. 미지의 값을 대비한 인코더 학습

le = LabelEncoder()
values = X[col].astype(str).tolist()
if "unknown" not in values:
    values.append("unknown") # unknown 범주 강제 추가
le.fit(values)

# 4. 새롭게 알게 된 내용/ 어려운 내용/ 배울 점
- 대규모 결측치의 범주화: 90%가 넘는 결측치를 삭제하지 않고 "unknown"으로 채워 넣었음. 데이터가 많을 때는 결측 자체가 하나의 중요한 신호(Signal)가 될 수 있다는 점을 알 수 있었음
- MCC 지표: Matthew's Correlation Coefficient라는 예측력 평가 지표로, 불균형 데이터나 대규모 분류 문제에서 정확도보다 더 신뢰할 수 있는 지표라는 것을 학습함
- 인코딩 로직: if x in le.classes_ 조건문을 사용해 테스트 데이터의 예외 상황을 처리한 코드를 학습함